[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch01/NN_DL_ch01_CreditMLP/NN_DL_ch01_CreditMLP.ipynb)

# NN_DL_ch01_CreditMLP

**MLP for Credit Scoring using the German Credit Dataset**

We load the real German Credit dataset from OpenML, preprocess it, train a multi-layer perceptron in PyTorch, and evaluate with ROC curves, confusion matrices, and feature importance analysis.

In [ ]:
!pip install -q torch scikit-learn matplotlib

In [ ]:
# Style & Reproducibility
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MAIN_BLUE = '#1A3A6E'
IDA_RED   = '#CD0000'
FOREST    = '#2E7D32'
CRIMSON   = '#DC3545'
AMBER     = '#FFC107'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 14,
    'figure.figsize': (10, 6),
})
np.random.seed(42)

def save_fig(name):
    plt.savefig(f'{name}.pdf', bbox_inches='tight', dpi=300, transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', dpi=150, transparent=True)
    plt.show()
    print(f'Saved {name}.pdf and {name}.png')

In [ ]:
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load German Credit Data
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd

data = fetch_openml('credit-g', version=1, as_frame=True, parser='auto')
df = data.frame
print(f"Dataset shape: {df.shape}")
print(f"Target distribution:\n{df['class'].value_counts()}")

# Encode categoricals
le_dict = {}
X = df.drop('class', axis=1).copy()
for col in X.select_dtypes(include='category').columns:
    le_dict[col] = LabelEncoder()
    X[col] = le_dict[col].fit_transform(X[col].astype(str))

y = (df['class'] == 'bad').astype(int).values
feature_names = list(X.columns)
X = X.values.astype(np.float32)

scaler = StandardScaler()
X = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"\nTrain: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Bad credit rate: train={y_train.mean():.3f}, test={y_test.mean():.3f}")

In [ ]:
# MLP Model
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class CreditMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1))
    def forward(self, x):
        return self.net(x)

model = CreditMLP(X_train.shape[1]).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training
pos_weight = torch.tensor([(1 - y_train.mean()) / y_train.mean()]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
test_ds  = TensorDataset(torch.FloatTensor(X_test),  torch.FloatTensor(y_test))
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=256)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(100):
    model.train()
    losses, correct, total = [], 0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb).squeeze()
        loss = criterion(logits, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item())
        correct += ((logits > 0).float() == yb).sum().item()
        total += len(yb)
    history['train_loss'].append(np.mean(losses))
    history['train_acc'].append(correct / total)

    model.eval()
    losses, correct, total = [], 0, 0
    with torch.no_grad():
        for xb, yb in test_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb).squeeze()
            losses.append(criterion(logits, yb).item())
            correct += ((logits > 0).float() == yb).sum().item()
            total += len(yb)
    history['val_loss'].append(np.mean(losses))
    history['val_acc'].append(correct / total)
    scheduler.step(history['val_loss'][-1])

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss {history['train_loss'][-1]:.4f} "
              f"Acc {history['train_acc'][-1]:.3f} | Val Loss {history['val_loss'][-1]:.4f} "
              f"Acc {history['val_acc'][-1]:.3f}")

In [ ]:
# Plot Training Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], color=MAIN_BLUE, label='Train Loss', lw=2)
ax1.plot(history['val_loss'],   color=IDA_RED,   label='Val Loss',   lw=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss'); ax1.legend()

ax2.plot(history['train_acc'], color=MAIN_BLUE, label='Train Acc', lw=2)
ax2.plot(history['val_acc'],   color=IDA_RED,   label='Val Acc',   lw=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Accuracy'); ax2.legend()

plt.suptitle('Credit MLP Training History', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch01_credit_training')

In [ ]:
# ROC Curve
from sklearn.metrics import roc_curve, auc, classification_report

model.eval()
with torch.no_grad():
    y_score = torch.sigmoid(model(torch.FloatTensor(X_test).to(device))).cpu().numpy().ravel()
y_pred = (y_score > 0.5).astype(int)

fpr, tpr, _ = roc_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, color=MAIN_BLUE, lw=2.5, label=f'MLP (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='grey', ls='--', lw=1, label='Random')
ax.fill_between(fpr, tpr, alpha=0.15, color=MAIN_BLUE)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve - Credit Scoring MLP', fontweight='bold')
ax.legend(loc='lower right', fontsize=12)
save_fig('nn_ch01_credit_roc')

print(classification_report(y_test, y_pred, target_names=['Good', 'Bad']))

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=['Good Credit', 'Bad Credit'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title('Confusion Matrix - Credit Scoring MLP', fontweight='bold')
save_fig('nn_ch01_credit_confusion')

In [ ]:
# Feature Importance (Permutation)
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator, ClassifierMixin

class TorchWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, m, dev):
        self.m = m; self.dev = dev
    def fit(self, X, y): return self
    def predict(self, X):
        self.m.eval()
        with torch.no_grad():
            return (self.m(torch.FloatTensor(X).to(self.dev)).cpu().numpy().ravel() > 0).astype(int)
    def predict_proba(self, X):
        self.m.eval()
        with torch.no_grad():
            p = torch.sigmoid(self.m(torch.FloatTensor(X).to(self.dev))).cpu().numpy().ravel()
        return np.column_stack([1 - p, p])

wrapper = TorchWrapper(model, device)
perm = permutation_importance(wrapper, X_test, y_test, n_repeats=10,
                              random_state=42, scoring='roc_auc')

idx = perm.importances_mean.argsort()[-15:]
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(idx)), perm.importances_mean[idx], color=MAIN_BLUE, edgecolor='white')
ax.set_yticks(range(len(idx)))
ax.set_yticklabels([feature_names[i] for i in idx])
ax.set_xlabel('Mean Permutation Importance (AUC decrease)')
ax.set_title('Top 15 Feature Importances - Credit MLP', fontweight='bold')
plt.tight_layout()
save_fig('nn_ch01_credit_importance')

## Summary

Trained an MLP on the **German Credit dataset** (1000 samples, 20 features) for credit scoring with class-weighted loss. Evaluated via ROC-AUC, confusion matrix, and permutation feature importance.